# 線性搜尋 vs. 二分搜尋

> 目標：透過程式與圖像化追蹤，理解兩種搜尋策略的步驟與成本差異。


## 1. 線性搜尋回顧
線性搜尋會從索引 0 開始逐一比較。以下範例計算比較次數：


In [4]:
def linear_search(arr, target):
    steps = 0
    for index, value in enumerate(arr):
        steps += 1
        if value == target:
            return index, steps
    return -1, steps

data = list(range(0, 101, 5))
print(linear_search(data, 45))
print(linear_search(data, 102))


(9, 10)
(-1, 21)


### 線性搜尋更多案例
- 觀察目標值在陣列不同位置時所需的比較次數。
- 嘗試插入不存在的目標，體會最壞情況的成本。


In [5]:
targets = [data[0], data[len(data) // 2], data[-1], 102]
for t in targets:
    index, steps = linear_search(data, t)
    status = "找到" if index != -1 else "未找到"
    position = f"{index:>3}" if index != -1 else "--"
    print(f"目標 {t:>3} -> {status} | 索引 {position:>3} | 步數 {steps:>2}")


目標   0 -> 找到 | 索引   0 | 步數  1
目標  50 -> 找到 | 索引  10 | 步數 11
目標 100 -> 找到 | 索引  20 | 步數 21
目標 102 -> 未找到 | 索引  -- | 步數 21


> 最壞情況下，線性搜尋需要掃描完整陣列，步數約等於元素個數。


## 2. 二分搜尋示意
在排序陣列上使用二分搜尋，記錄 `low`, `mid`, `high` 的變化：


In [6]:
def binary_search(arr, target):
    low, high = 0, len(arr) - 1
    steps = 0
    trace = []
    while low <= high:
        steps += 1
        mid = (low + high) // 2
        trace.append((low, mid, high))
        if arr[mid] == target:
            return mid, steps, trace
        elif arr[mid] < target:
            low = mid + 1
        else:
            high = mid - 1
    return -1, steps, trace

result = binary_search(data, 45)
print('查得索引:', result[0], '| 次數:', result[1])
print('追蹤記錄 (low, mid, high):', result[2])


查得索引: 9 | 次數: 5
追蹤記錄 (low, mid, high): [(0, 10, 20), (0, 4, 9), (5, 7, 9), (8, 8, 9), (9, 9, 9)]


### 二分搜尋更多情境
- 追蹤區間的縮小過程，分辨成功與失敗的行為差異。
- 注意 `trace` 最後一次紀錄代表哪一個搜尋範圍。


In [7]:
test_targets = [data[0], data[len(data) // 2], data[-1], 102]
for t in test_targets:
    index, steps, trace = binary_search(data, t)
    status = "找到" if index != -1 else "未找到"
    print(f"目標 {t:>3} -> {status} | 步數 {steps}")
    for step, (low, mid, high) in enumerate(trace, start=1):
        print(f"  step {step:>2}: low={low}, mid={mid}, high={high}")
    print()


目標   0 -> 找到 | 步數 4
  step  1: low=0, mid=10, high=20
  step  2: low=0, mid=4, high=9
  step  3: low=0, mid=1, high=3
  step  4: low=0, mid=0, high=0

目標  50 -> 找到 | 步數 1
  step  1: low=0, mid=10, high=20

目標 100 -> 找到 | 步數 5
  step  1: low=0, mid=10, high=20
  step  2: low=11, mid=15, high=20
  step  3: low=16, mid=18, high=20
  step  4: low=19, mid=19, high=20
  step  5: low=20, mid=20, high=20

目標 102 -> 未找到 | 步數 5
  step  1: low=0, mid=10, high=20
  step  2: low=11, mid=15, high=20
  step  3: low=16, mid=18, high=20
  step  4: low=19, mid=19, high=20
  step  5: low=20, mid=20, high=20



> 當目標不存在時，`low` 會超過 `high`，表示搜尋範圍已被縮小到空集合。


### 可視化建議
- 將 `trace` 用圖表或列表呈現。
- 請學生手動畫出搜尋範圍縮小過程，確認理解。


### 平均步數觀察
透過隨機實驗快速估計兩種搜尋策略的平均比較次數：


In [8]:
import random

random.seed(0)

def average_steps(array_length, trials=20):
    arr = list(range(0, array_length * 2, 2))
    linear_total = 0
    binary_total = 0
    choices = arr + [-1, array_length * 2 + 1]
    for _ in range(trials):
        target = random.choice(choices)
        linear_total += linear_search(arr, target)[1]
        binary_total += binary_search(arr, target)[1]
    return linear_total / trials, binary_total / trials

for n in [10, 40, 160, 320]:
    linear_avg, binary_avg = average_steps(n)
    print(f"n={n:>3} -> 線性平均 {linear_avg:>5.2f} | 二分平均 {binary_avg:>5.2f}")


n= 10 -> 線性平均  6.15 | 二分平均  2.75
n= 40 -> 線性平均 24.50 | 二分平均  4.55
n=160 -> 線性平均 79.45 | 二分平均  6.70
n=320 -> 線性平均 181.05 | 二分平均  7.60


> 隨著陣列變長，二分搜尋的平均步數成長趨緩，適合大型資料集。


## 3. 練習題
1. 請修改二分搜尋函式，在每次迴圈印出 `low`, `mid`, `high`。
2. 分別搜尋陣列中最小值、最大值、不存在的值，觀察步數並解釋原因。
3. 仿照上方範例蒐集 `trace`，將每一步的搜尋範圍可視化（列表或圖表）。
4. 以不同陣列長度（例如 20、100、500）重複實驗，整理線性與二分搜尋的平均步數。
5. 撰寫函式輸入多個 `target`，比較線性與二分搜尋的平均比較次數並分析差異。


In [9]:
# TODO：在此區塊進行你的實驗，並記錄觀察結果
# 建議紀錄內容：搜尋目標、比較次數、搜尋順序、結論摘要



### 延伸挑戰
- 將二分搜尋改寫成遞迴版本，並比較遞迴深度與步數。
- 對未排序陣列先排序，再搜尋，計算整體成本。
- 在存在重複值的情況下，修改二分搜尋回傳所有匹配索引或插入點範圍。
- 參考程式語言內建函式（如 Python `bisect`）驗證你的實作行為。
